# Altair Tutorial. Part 3. Interaction
## Hands-on practical work

In [ ]:
!pip install altair==6.0.0
!pip install pandas
!pip install vega_datasets


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Using cached vega_datasets-0.9.0-py3-none-any.whl.metadata (5.5 kB)
Using cached vega_datasets-0.9.0-py3-none-any.whl (210 kB)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# Interaction
We can create interactive charts also in Altair. And add widgets to interact with the data.
To add interactions, several aspects can be included:
- Conditions: let the chart to change according to the data, and if the data changes (e.g., by filtering the data differently), the chart will change
- Sliders: Can be used to select values (that can then be employed to filter data)
- Dropdowns: They can also be used to select data.
- Radio buttons and checkboxes: Same as before
- Direct interactions: Can be used to select data that then can be included in filters or conditions
- Zoom and pan: Can be used to explore the data

## Interactive chart

Can be simply implemented by adding the *interactive* call to the chart

In [8]:
#@title Interactive fertility vs life expectancy scatterplot of two countries
import altair as alt
import pandas as pd
from vega_datasets import data
df = data.gapminder()

alt.Chart(df).mark_circle().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Color('country:N'),
    opacity = 'year:O'
).transform_filter((alt.datum.year > 1980)
& ((alt.datum.country == 'Afghanistan')   # We select Afghanistan
| (alt.datum.country == 'Argentina'))     # and Argentina
).interactive()

# Drag with the mouse and zoom

alt.Chart(...)

## Slider
A Slider is defined using the *alt.binding_range* function, together with some condition or filter that makes the slider act upon the data.

In [10]:
#@title Interactive fertility vs life expectancy scatterplot of two countries
import altair as alt
import pandas as pd
from vega_datasets import data
df = data.gapminder()

slider = alt.binding_range(min=1960, max=2005, step=5, name='year:')
op_var = alt.param(value=1960, bind=slider)

alt.Chart(df).mark_circle().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Color('country:N'),
    opacity = 'year:O'
).transform_filter((alt.datum.year > op_var)   # Filter data based on condition
& ((alt.datum.country == 'Afghanistan')
| (alt.datum.country == 'Argentina'))).add_params(op_var) # Pass the parameter


alt.Chart(...)

## Changing opacity

The parameter can be used to define variables, such as opacity

In [11]:
#@title Interactive fertility vs life expectancy scatterplot of two countries
import altair as alt
import pandas as pd
from vega_datasets import data

df = data.gapminder()

slider = alt.binding_range(min=0, max=1, step=0.05, name='opacity:')
op_var = alt.param(value=0.1, bind=slider)

alt.Chart(df).mark_circle(opacity=op_var).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Color('country:N'),
).transform_filter((alt.datum.year > 1960)
& ((alt.datum.country == 'Afghanistan')
| (alt.datum.country == 'Argentina'))).add_params(op_var)

alt.Chart(...)

## Radio button

Radio buttons work in a similar way

In [ ]:
#@title Radio button example
import altair as alt
from vega_datasets import data
import pandas as pd

df = data.cars()

radio_button = alt.binding_radio(labels = ['Europe', 'USA', 'Japan'],
                                 options = ['Europe', 'USA', 'Japan'],
                                 name = 'Country')

origin_par = alt.param(value = 'Europe', bind = radio_button)

alt.Chart(df).mark_point().encode(
    alt.X('Cylinders:Q'),
    alt.Y('Horsepower:Q'),
    alt.Color('Origin:N')
).transform_filter(alt.datum.Origin == origin_par).add_params(origin_par)


alt.Chart(...)

Some things are wrong with the previous chart, such as the fact that the scales change if we change the option. This should be fixed. And the colors of the cars should be probably dependent on the country.

## Selections

Altair also facilitates the selection of elements, with the *alt.selection_point* option. In the following example, we select a point and make it opaque, while the non selected ones are made more transparent.

In [12]:
#@title Selection example
import altair as alt
from vega_datasets import data
import pandas as pd

df = data.gapminder()

select_point = alt.selection_point(nearest = True)

alt.Chart(df).mark_circle().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Color('country:N'),
    opacity = alt.condition(select_point, alt.value(1.0), alt.value(0.2))
).transform_filter((alt.datum.year > 1980)
& ((alt.datum.country == 'Afghanistan')
| (alt.datum.country == 'Argentina'))).add_params(select_point)


alt.Chart(...)

## Hover

We can make the selection to happen over hovering


In [ ]:
#@title Selection example
import altair as alt
from vega_datasets import data
import pandas as pd

df = data.gapminder()

select_point = alt.selection_point(on='mouseover', nearest = True) # Mouseover

alt.Chart(df).mark_circle().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Color('country:N'),
    opacity = alt.condition(select_point, alt.value(1.0), alt.value(0.2))
).transform_filter((alt.datum.year > 1980)
& ((alt.datum.country == 'Afghanistan')
| (alt.datum.country == 'Argentina'))).add_params(select_point)


alt.Chart(...)

## Cross selections

Things begin to be interesting when a selection in a chart, makes modifications to another chart. This is called cross interaction.
It is also easy to do in altair.

In [1]:
#@title Cross highlighting
import altair as alt
from vega_datasets import data
import pandas as pd

df = data.gapminder()

select_interval = alt.selection_interval(empty = False)

ch1 = alt.Chart(df).mark_circle().encode(
    alt.X('life_expect:Q', title = 'Life expectancy'),
    alt.Y('pop:Q', title = 'Population'),
    color = alt.condition(select_interval, 'country:N', alt.value('lightgray'))
).transform_filter((alt.datum.year > 1980)
& (alt.datum.pop > 100000000)
).add_params(select_interval)

ch2 = alt.Chart(df).mark_circle().encode(
    alt.X('life_expect:Q', title = 'Life expectancy'),
    alt.Y('fertility:Q', title = 'Fertility'),
    color = alt.condition(select_interval, 'country:N', alt.value('lightgray'))
).transform_filter((alt.datum.year > 1980)
& (alt.datum.pop > 100000000)
).add_params(select_interval)

ch1 | ch2

/var/folders/v5/jx4rcwvs44bd_b45d0yvr_8w0000gn/T/ipykernel_18080/801063569.py:26: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  ch1 | ch2


alt.HConcatChart(...)

In [2]:
import altair as alt
from altair.datasets import data

source = alt.UrlData(
    data.flights_2k.url,
    format={'parse': {'date': 'date'}}
)

brush = alt.selection_interval(encodings=['x'])

# Define the base chart, with the common parts of the
# background and highlights
base = alt.Chart(width=160, height=130).mark_bar().encode(
    x=alt.X(alt.repeat('column')).bin(maxbins=20),
    y='count()'
)

# gray background with selection
background = base.encode(
    color=alt.value('#ddd')
).add_params(brush)

# blue highlights on the transformed data
highlight = base.transform_filter(brush)

# layer the two charts & repeat
alt.layer(
    background,
    highlight,
    data=source
).transform_calculate(
    "time",
    "hours(datum.date)"
).repeat(column=["distance", "delay", "time"])

alt.RepeatChart(...)

## Selections in a single dimension

Selections can be done for rectangular regions, points, but also for a single dimension. And they can perform several modifications, such as changing the size of the selected objects, as in the following example.


In [ ]:
#@title Cross highlighting
import altair as alt
from vega_datasets import data
import pandas as pd

df = data.gapminder()

select_interval = alt.selection_interval(empty = False, encodings = ['y'])

ch1 = alt.Chart(df).mark_circle().encode(
    alt.X('life_expect:Q', title = 'Life expectancy'),
    alt.Y('pop:Q', title = 'Population'),
    color = alt.condition(select_interval, alt.value('crimson'),
                          'country:N'),
).transform_filter((alt.datum.year > 1980)
& (alt.datum.pop > 100000000)
).add_params(select_interval)

ch2 = alt.Chart(df).mark_circle().encode(
    alt.X('life_expect:Q', title = 'Life expectancy'),
    alt.Y('fertility:Q', title = 'Fertility'),
    color = alt.condition(select_interval, alt.value('crimson'),
                          'country:N'),
    size = alt.condition(select_interval, alt.value(60), alt.value(30))
).transform_filter((alt.datum.year > 1980)
& (alt.datum.pop > 100000000)
).add_params(select_interval)

ch1 | ch2

/tmp/ipykernel_513/2183638989.py:29: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  ch1 | ch2


alt.HConcatChart(...)

## Interactive legends

We can also have interactive legends in the form of another chart.

In [ ]:
#@title Interactive legend

import altair as alt
from vega_datasets import data
import pandas as pd

df = data.cars()

select_point = alt.selection_point(encodings = ['color'])

color = alt.condition(select_point,
                      alt.Color('Origin:N', legend = None),
                      alt.value('lightgray'))

scatter = alt.Chart(df).mark_point().encode(
    alt.X('Horsepower:Q'),
    alt.Y('Miles_per_Gallon:Q').title('Miles per gallon'),
    color = color,
    tooltip = 'Name:N'
).add_params(select_point)

legend = alt.Chart(df).mark_circle().encode(
    alt.Y('Origin:N').axis(orient='right'),
    color = color,
).add_params(select_point)

scatter | legend

/tmp/ipykernel_513/2731904803.py:27: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  scatter | legend


alt.HConcatChart(...)

In [ ]:
#@title Interactive legend

import altair as alt
from vega_datasets import data
import pandas as pd

df = data.cars()

select_point = alt.selection_point(fields = ['Origin', 'Cylinders'])

color = alt.condition(select_point,
                      alt.Color('Origin:N', legend = None),
                      alt.value('lightgray'))

scatter = alt.Chart(df).mark_point().encode(
    alt.X('Horsepower:Q'),
    alt.Y('Miles_per_Gallon:Q').title('Miles per gallon'),
    color = color,
    tooltip = 'Name:N'
).add_params(select_point)

legend = alt.Chart(df).mark_rect().encode(
    alt.X('Cylinders:O'),
    alt.Y('Origin:N').axis(orient='right'),
    color = color,
).add_params(select_point)

scatter | legend

/tmp/ipykernel_513/2234989530.py:28: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  scatter | legend


alt.HConcatChart(...)

We can have multiple selections in our charts.

In [ ]:
#@title Multiple selections

import altair as alt
from vega_datasets import data
import pandas as pd

source = data.gapminder()

clusters = pd.DataFrame([
    {"id": 0, "name": "South Asia"},
    {"id": 1, "name": "Europe & Central Asia"},
    {"id": 2, "name": "Sub-Saharan Africa"},
    {"id": 3, "name": "America"},
    {"id": 4, "name": "East Asia & Pacific"},
    {"id": 5, "name": "Middle East & North Africa"},

])

cluster_dropdown = alt.binding_select(
    options = ['South Asia', 'Europe & Central Asia', 'Sub-Saharan Africa',
               'America', 'East Asia & Pacific', 'Middle East & North Africa'
    ]
)

dropSelect = alt.selection_point(fields = ['name'],
                                 bind = cluster_dropdown, name = 'Region')

input_year = alt.binding_range(min = 1960, max = 2005, step = 5)

yearSelect = alt.selection_point(fields = ['year'], bind = input_year,
                                 name = 'Year:')

alt.Chart(source).mark_circle(opacity = 1.0).encode(
    alt.X('fertility:Q', title = 'Fertility'),
    alt.Y('life_expect:Q', title = 'Life Expectancy'),
    color = alt.condition(dropSelect, 'name:N', alt.value('lightgray')),
    opacity = alt.condition(yearSelect, alt.value(1.0), alt.value(0.0))
).transform_lookup(
    lookup = 'cluster',
    from_ = alt.LookupData(
        data = clusters, key = 'id', fields = ['name']
    )
).add_params(dropSelect, yearSelect)



alt.Chart(...)

## Overview and detail

Overview and detail can be created using the same or different charts

In [ ]:
#@title Overview and detail
import altair as alt
import pandas as pd

from vega_datasets import data

source = data.flights_5k.url

brush = alt.selection_interval(encodings = ['x'])

base = alt.Chart(source).transform_calculate(
    time = 'hours(datum.date) + minutes(datum.date) / 60'
).mark_bar().encode(alt.Y('count():Q')
).properties( width = 600, height = 100)

detail = base.encode(
    alt.X('time:Q', bin = alt.Bin(maxbins = 30, extent = brush),
    scale = alt.Scale(domain = brush)
  )
)

overview = base.encode(alt.X('time:Q', bin = alt.Bin(maxbins = 30)),
  ).add_params(brush)

alt.vconcat(detail, overview)

# Drag over the bottom chart

alt.VConcatChart(...)

## Interactive details on demand

In [ ]:
#@title The rule bar appears on hovering, as well as the labels of the data
import altair as alt
import pandas as pd
import numpy as np

np.random.seed(42)
source = pd.DataFrame(
    np.cumsum(np.random.randn(100, 3), 0).round(2),
    columns=['A', 'B', 'C'], index=pd.RangeIndex(100, name='x')
)
source = source.reset_index().melt('x', var_name='category', value_name='y')

# Create a selection that chooses the nearest point & selects based on x-value
nearest = alt.selection_point(nearest=True, on='mouseover',
                        fields=['x'], empty=False)

# The basic line
line = alt.Chart(source).mark_line(interpolate='basis').encode(
    x='x:Q',
    y='y:Q',
    color='category:N'
)

# Transparent selectors across the chart. This is what tells us
# the x-value of the cursor
selectors = alt.Chart(source).mark_point().encode(
    x='x:Q',
    opacity=alt.value(0),
).add_params(
    nearest
)

# Draw points on the line, and highlight based on selection
points = line.mark_point().encode(
    opacity=alt.condition(nearest, alt.value(1), alt.value(0))
)

# Draw text labels near the points, and highlight based on selection
text = line.mark_text(align='left', dx=5, dy=-5).encode(
    text=alt.condition(nearest, 'y:Q', alt.value(' '))
)

# Draw a rule at the location of the selection
rules = alt.Chart(source).mark_rule(color='gray').encode(
    x='x:Q',
).transform_filter(
    nearest
)

# Put the five layers into a chart and bind the data
alt.layer(
    line, selectors, points, rules, text
).properties(
    width=600, height=300
)

alt.LayerChart(...)

In [ ]:
#@title Example from Altair library imitating Gapminders' Trendalizer

import altair as alt
from vega_datasets import data

# Data source
source = data.gapminder.url

# X-value slider
x_slider = alt.binding_range(min=1955, max=2005, step=5, name='Year ')
x_select = alt.selection_point(name="x_select", fields=['year'], bind=x_slider, value=1980)

# Hover selection
hover = alt.selection_point(on='mouseover', fields=['country'], empty=False)
# A separate hover for the points since these need empty=True
hover_point_opacity = alt.selection_point(on='mouseover', fields=['country'])

# Search box for country name
search_box = alt.param(
    value='',
    bind=alt.binding(input='search', placeholder="Country", name='Search ')
)

# Base chart
base = alt.Chart(source).encode(
    x=alt.X('fertility:Q', scale=alt.Scale(zero=False), title='Babies per woman (total fertility rate)'),
    y=alt.Y('life_expect:Q', scale=alt.Scale(zero=False), title='Life expectancy'),
    color=alt.Color('region:N', title='Region', legend=alt.Legend(orient='bottom-left', titleFontSize=14, labelFontSize=12), scale=alt.Scale(scheme='dark2')),
    detail='country:N'
).transform_calculate(
    region="""{
        '0': 'South Asia',
        '1': 'Europe & Central Asia',
        '2': 'Sub-Saharan Africa',
        '3': 'The Americas',
        '4': 'East Asia & Pacific',
        '5': 'Middle East & North Africa'
    }[datum.cluster]"""
).transform_filter(
    # Exclude North Korea and South Korea due to source data error
    "datum.country !== 'North Korea' && datum.country !== 'South Korea'"
)

search_matches = alt.expr.test(alt.expr.regexp(search_box, "i"), alt.datum.country)
opacity = (
    alt.when(hover_point_opacity, search_matches)
    .then(alt.value(0.8))
    .otherwise(alt.value(0.1))
)
# Points that are always visible (filtered by slider and search)
visible_points = base.mark_circle(size=100).encode(
    opacity=opacity
    ).transform_filter(
        x_select
    ).add_params(
        hover,
        hover_point_opacity,
        x_select
)

when_hover = alt.when(hover)
hover_line = alt.layer(
    # Line layer
    base.mark_trail().encode(
        order=alt.Order(
            'year:Q',
            sort='ascending'
        ),
        size=alt.Size(
            'year:Q',
            scale=alt.Scale(domain=[1955, 2005], range=[1, 12]),
            legend=None
        ),
        opacity=when_hover.then(alt.value(0.3)).otherwise(alt.value(0)),
        color=alt.value('#222222')
    ),
    # Point layer
    base.mark_point(size=50).encode(
        opacity=when_hover.then(alt.value(0.8)).otherwise(alt.value(0)),
    )
)

# Year labels
year_labels = base.mark_text(align='left', dx=5, dy=-5, fontSize=14).encode(
    text='year:O',
    color=alt.value('#222222')
).transform_filter(hover)

# Country labels
country_labels = alt.Chart(source).mark_text(
    align='left',
    dx=-15,
    dy=-25,
    fontSize=18,
    fontWeight='bold'
).encode(
    x='fertility:Q',
    y='life_expect:Q',
    text='country:N',
    color=alt.value('black'),
    opacity=when_hover.then(alt.value(1)).otherwise(alt.value(0))
).transform_window(
    rank='rank(life_expect)',
    sort=[alt.SortField('life_expect', order='descending')],
    groupby=['country'] # places label atop highest point on y-axis on hover
).transform_filter(
    alt.datum.rank == 1
).transform_aggregate(
    life_expect='max(life_expect)',
    fertility='max(fertility)',
    groupby=['country']
)

background_year = alt.Chart(source).mark_text(
    baseline='middle',
    fontSize=96,
    opacity=0.2
).encode(
    text='year:O'
).transform_filter(
    x_select
).transform_aggregate(
    year='max(year)'
)

# Combine all layers
chart = alt.layer(
    visible_points, year_labels, country_labels, hover_line, background_year
).properties(
    width=500,
    height=500,
    padding=10  # Padding ensures labels fit
).configure_axis(
    labelFontSize=12,
    titleFontSize=12
).add_params(search_box)

chart

alt.LayerChart(...)

In [ ]:
#@title Using VegaFusion to accelerate interactions with large datasets

!pip install "vegafusion[embed]>=1.5.0" "vegafusion>=1.5.0"
!pip install "vl-convert-python>=1.6.0"




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 14.9 MB/s eta 0:00:00


In [ ]:

#@title Example from Altair library imitating Gapminders' Trendalizer, now with a dual axis chart with GDP and CO2 per capita.

import altair as alt
from vega_datasets import data

# Data source
source = data.gapminder.url

# X-value slider
x_slider = alt.binding_range(min=1955, max=2005, step=5, name='Year ')
x_select = alt.selection_point(name="x_select", fields=['year'], bind=x_slider, value=1980)

# Hover selection
hover = alt.selection_point(on='mouseover', fields=['country'], empty=False)
# A separate hover for the points since these need empty=True
hover_point_opacity = alt.selection_point(on='mouseover', fields=['country'])

# Search box for country name
search_box = alt.param(
    value='',
    bind=alt.binding(input='search', placeholder="Country", name='Search ')
)

# Base chart
base = alt.Chart(source).encode(
    x=alt.X('fertility:Q', scale=alt.Scale(zero=False), title='Babies per woman (total fertility rate)'),
    y=alt.Y('life_expect:Q', scale=alt.Scale(zero=False), title='Life expectancy'),
    color=alt.Color('region:N', title='Region', legend=alt.Legend(orient='bottom-left', titleFontSize=14, labelFontSize=12), scale=alt.Scale(scheme='dark2')),
    detail='country:N'
).transform_calculate(
    region="""{
        '0': 'South Asia',
        '1': 'Europe & Central Asia',
        '2': 'Sub-Saharan Africa',
        '3': 'The Americas',
        '4': 'East Asia & Pacific',
        '5': 'Middle East & North Africa'
    }[datum.cluster]"""
).transform_filter(
    # Exclude North Korea and South Korea due to source data error
    "datum.country !== 'North Korea' && datum.country !== 'South Korea'"
)

search_matches = alt.expr.test(alt.expr.regexp(search_box, "i"), alt.datum.country)
opacity = (
    alt.when(hover_point_opacity, search_matches)
    .then(alt.value(0.8))
    .otherwise(alt.value(0.1))
)
# Points that are always visible (filtered by slider and search)
visible_points = base.mark_circle(size=100).encode(
    opacity=opacity
    ).transform_filter(
        x_select
    ).add_params(
        hover,
        hover_point_opacity,
        x_select
)

when_hover = alt.when(hover)
hover_line = alt.layer(
    # Line layer
    base.mark_trail().encode(
        order=alt.Order(
            'year:Q',
            sort='ascending'
        ),
        size=alt.Size(
            'year:Q',
            scale=alt.Scale(domain=[1955, 2005], range=[1, 12]),
            legend=None
        ),
        opacity=when_hover.then(alt.value(0.3)).otherwise(alt.value(0)),
        color=alt.value('#222222')
    ),
    # Point layer
    base.mark_point(size=50).encode(
        opacity=when_hover.then(alt.value(0.8)).otherwise(alt.value(0)),
    )
)

# Year labels
year_labels = base.mark_text(align='left', dx=5, dy=-5, fontSize=14).encode(
    text='year:O',
    color=alt.value('#222222')
).transform_filter(hover)

# Country labels
country_labels = alt.Chart(source).mark_text(
    align='left',
    dx=-15,
    dy=-25,
    fontSize=18,
    fontWeight='bold'
).encode(
    x='fertility:Q',
    y='life_expect:Q',
    text='country:N',
    color=alt.value('black'),
    opacity=when_hover.then(alt.value(1)).otherwise(alt.value(0))
).transform_window(
    rank='rank(life_expect)',
    sort=[alt.SortField('life_expect', order='descending')],
    groupby=['country'] # places label atop highest point on y-axis on hover
).transform_filter(
    alt.datum.rank == 1
).transform_aggregate(
    life_expect='max(life_expect)',
    fertility='max(fertility)',
    groupby=['country']
)

background_year = alt.Chart(source).mark_text(
    baseline='middle',
    fontSize=96,
    opacity=0.2
).encode(
    text='year:O'
).transform_filter(
    x_select
).transform_aggregate(
    year='max(year)'
)

# Combine all layers
chart = alt.layer(
    visible_points, year_labels, country_labels, hover_line, background_year
).properties(
    width=500,
    height=500,
    #padding=10  # Padding ensures labels fit -> need to remove for the concat (also removed config_axis)
).add_params(search_box)



# Adding new data from a public source
owid_url = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"

right_base = (
    alt.Chart(owid_url, title="Selected country: GDP per capita & CO₂ per capita")
      .transform_filter(hover)
      .transform_filter("datum.year >= 1960 && isValid(datum.population) && isValid(datum.gdp) && isValid(datum.co2_per_capita)")
      .transform_calculate(gdp_per_capita="datum.gdp / datum.population")
      .properties(width=500, height=500)
)

gdp_line = (
    right_base.mark_line()
      .encode(
          x=alt.X("year:Q", axis=alt.Axis(title=None, tickMinStep=10)),
          y=alt.Y("gdp_per_capita:Q", axis=alt.Axis(title="GDP per capita (current $)"),
                  scale=alt.Scale(zero=False)),
          color=alt.value("#1f77b4"),
          tooltip=["country:N","year:Q","gdp_per_capita:Q"]
      )
)

co2_line = (
    right_base.mark_line(strokeDash=[4,3])
      .encode(
          x="year:Q",
          y=alt.Y("co2_per_capita:Q",
                  axis=alt.Axis(title="CO₂ per capita (t)", orient="right"),
                  scale=alt.Scale(zero=False)),
          color=alt.value("#d62728"),
          tooltip=["country:N","year:Q","co2_per_capita:Q"]
      )
)

right_panel = (gdp_line + co2_line).resolve_scale(y="independent")

final = (
    alt.hconcat(chart, right_panel)
       .properties(padding=10)
       .configure_axis(labelFontSize=12, titleFontSize=12)
       .configure_view(stroke=None)
)
final

alt.HConcatChart(...)